In [ ]:
# =============================================================================
# VALIDATION SET BUILDER — random stratified lines with full scene context
# =============================================================================
# 15 random lines per film, stratified across the full film by position.
# Songs are excluded when possible. Each line includes the full scene text
# as context (the same input the LLM receives for addressee tagging).
#
# Annotation columns (12 total, 4 labels × 3 raters):
#   addressee_*      : character_id of the person being spoken to
#   addressee_type_* : individual / monologue / group / unclear
#   emotion_label_*  : anger / disgust / fear / joy / neutral / sadness / surprise
#   sentiment_*      : +1 / 0 / -1
#
# Outputs saved to data/03_validation_dataset/addressee_validation_input/:
#   validation_for_annotation.csv   — no model columns (raters fill this in)
#   validation_with_predictions.csv — includes model columns for post-annotation analysis
# =============================================================================

import random
import pandas as pd
from pathlib import Path

SEED = 42
random.seed(SEED)

PROCESSED_DIR = Path("../data/02_processed")
OUTPUT_DIR = Path("../data/03_validation_dataset/addressee_validation_input")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_LINES = 15
SKIP = {"OLD_DATA", "beautyandthebeast"}

BASE_COLS = ["film_id", "scene_id", "utterance_id", "speaker",
             "canonical_name", "gender", "text", "word_count", "utterance_type"]
ANNOTATION_COLS = [
    "addressee_majo",      "addressee_sonia",      "addressee_oceane",
    "addressee_type_majo", "addressee_type_sonia", "addressee_type_oceane",
    "emotion_label_majo",  "emotion_label_sonia",  "emotion_label_oceane",
    "sentiment_majo",      "sentiment_sonia",       "sentiment_oceane",
]
PRED_COLS = ["addressee_llm", "addressee_type_llm",
             "emotion_label_model", "emotion_confidence_model", "sentiment_score_model"]


def best_file(film_dir: Path) -> Path | None:
    # Systematic source: utterances.csv is the §7 output of notebook 01,
    # enriched from the reviewed characters file (carries gender, plus the
    # correct underscore film_id/character_id). We deliberately do NOT use
    # utterances_with_addressee.csv: for some films it is a stale pre-review
    # artifact with a buggy hyphenated film_id and no gender column.
    c = film_dir / "utterances.csv"
    return c if c.exists() else None


def build_scene_text(scene_df: pd.DataFrame) -> str:
    sort_col = "line_number" if "line_number" in scene_df.columns else "utterance_id"
    lines = []
    for _, r in scene_df.sort_values(sort_col).iterrows():
        speaker = str(r.get("canonical_name") or r.get("speaker", "")).strip()
        text = str(r.get("text", "")).strip()
        if text:
            lines.append(f"{speaker}: {text}")
    return "\n".join(lines)


def sample_stratified(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    df = df.copy().reset_index(drop=True)
    if len(df) <= n:
        return df
    df["_bucket"] = pd.cut(range(len(df)), bins=n, labels=False)
    sampled = df.groupby("_bucket", group_keys=False).apply(
        lambda g: g.sample(1, random_state=seed)
    ).reset_index(drop=True)
    return sampled.drop(columns=["_bucket"], errors="ignore")


rows = []
print(f"{'Film':<42} lines")
print("-" * 52)

for film_dir in sorted(PROCESSED_DIR.iterdir()):
    if not film_dir.is_dir() or film_dir.name in SKIP:
        continue
    src = best_file(film_dir)
    if src is None:
        print(f"{film_dir.name:<42} no utterances file")
        continue

    df = pd.read_csv(src, low_memory=False, dtype=str).fillna("")

    # prefer spoken lines
    spoken = df[df["utterance_type"] != "song"] if "utterance_type" in df.columns else df
    if len(spoken) < N_LINES:
        spoken = df

    sampled = sample_stratified(spoken, N_LINES, SEED)

    scene_texts = {
        sid: build_scene_text(df[df["scene_id"] == sid])
        for sid in sampled["scene_id"].unique()
    }

    out = pd.DataFrame()
    for col in BASE_COLS:
        out[col] = sampled[col].values if col in sampled.columns else pd.NA
    out["scene_text"]         = sampled["scene_id"].map(scene_texts).values
    out["addressee_llm"]      = sampled["addressee"].values      if "addressee"      in sampled.columns else pd.NA
    out["addressee_type_llm"] = sampled["addressee_type"].values if "addressee_type" in sampled.columns else pd.NA
    for col in ["emotion_label_model", "emotion_confidence_model", "sentiment_score_model"]:
        out[col] = sampled[col].values if col in sampled.columns else pd.NA
    for col in ANNOTATION_COLS:
        out[col] = ""

    rows.append(out)
    print(f"{film_dir.name:<42} {len(out)}")

full_df = pd.concat(rows, ignore_index=True)
full_df[BASE_COLS + ["scene_text"] + ANNOTATION_COLS].to_csv(
    OUTPUT_DIR / "validation_for_annotation.csv", index=False)
full_df[BASE_COLS + ["scene_text"] + PRED_COLS + ANNOTATION_COLS].to_csv(
    OUTPUT_DIR / "validation_with_predictions.csv", index=False)

print(f"\nTotal: {len(full_df)} rows across {full_df['film_id'].nunique()} films")
print(f"Saved to: {OUTPUT_DIR.resolve()}")